In [1]:

!pip install langchain-groq langchain-community huggingface_hub PyPDF2 langchain-huggingface faiss-cpu

In [2]:
from dotenv import load_dotenv
from PyPDF2 import PdfReader
from langchain.text_splitter import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain.vectorstores import FAISS
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from huggingface_hub import login
import os
from langchain.prompts import PromptTemplate

In [3]:
!pip install ipywidgets

In [ ]:
load_dotenv()
login(os.getenv("HUGGING_FACE"))

In [5]:
!pip install python-dotenv

In [10]:
import os
print(os.getcwd())

c:\Users\Deepak Yadav\OneDrive\Desktop\project college


In [11]:
pdf_paths=['/content/medical.pdf']

In [12]:
def get_pdf_text(pdf_docs):
  text=""
  for pdf in pdf_docs:
    pdf_reader=PdfReader(pdf)
    for page in pdf_reader.pages:
      page_text=page.extract_text()
      if page_text:
        text+=page_text
  return text
raw_text=get_pdf_text(pdf_paths)
print("✅Extracted text length:",len(raw_text))

FileNotFoundError: [Errno 2] No such file or directory: '/content/medical.pdf'

In [ ]:
def get_text_chunks(raw_text):
  '''
  takes a single string of text and returns a list
  of text strings that can be fed to vector database
  '''
  text_splitter=CharacterTextSplitter(
      separator='\n',
      chunk_size=1000,
      chunk_overlap=200,
      length_function=len
  )
  chunks=text_splitter.split_text(raw_text)
  return chunks


In [ ]:
text_chunks=get_text_chunks(raw_text)
print("✅Number of text chunks:",len(text_chunks))

In [ ]:
def get_vectorstore(text_chunks):
  '''
  Creates a FAISS vector store from text chunks using embeddings
  '''
  embeddings=HuggingFaceEmbeddings(
      model_name="hkunlp/instructor-xl",
      model_kwargs={"device":"cuda"}

  )
  vectorstore=FAISS.from_texts(
      texts=text_chunks,
      embedding=embeddings
  )
  return vectorstore

vectorstore=get_vectorstore(text_chunks)
print("✅vectorstore created")

In [ ]:
db=vectorstore


In [ ]:
retriever=db.as_retriever

In [ ]:
template=""" You are a medical assistant.Using the following context,convert the medical report into a friendly summary for a patient.
Context:{context}
Medical Report:{question}
Summart:"""
prompt=PromptTemplate(
    input_variables=["context","question"],
    template=template,
)

In [ ]:
!pip install langchain_groq

In [ ]:
from langchain_groq import ChatGroq


In [ ]:
import os
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [ ]:
from langchain.chains import RetrievalQA

In [ ]:
qa_chain=RetrievalQA.from_chain_type(
    llm=llm,
    retriever=db.as_retriever(),
    chain_type_kwargs={"prompt":prompt}
)

**OUTPUT** **OF** **MEDICAL** **REPORT**

In [ ]:
query="summarize this medical report in simple terms"
response=qa_chain.run(query)
print("✅ Answer:\n",response)

In [ ]:
import streamlit as st